|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Sampling<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: a batched sampler you can prove is correct<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

Write the batched sampler. Give each request its own seed. Write a test that
can catch a wrong seed.

This is stage 13. You saw the performance part already. This part is about
correctness, which is harder to check. A broken sampler writes perfect
English.

In [ ]:
### run this cell

dev   = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB = 151936
torch.manual_seed(0)

def batch(B):
  return (torch.randn(B, VOCAB, device=dev),
          torch.rand(B, device=dev)*1.5 + 0.2,      # temperature
          torch.rand(B, device=dev)*0.3 + 0.7)      # top_p

print(f'vocabulary {VOCAB:,}')

# Exercise 1: one pass, one seed per request

`torch.multinomial` draws from a single global generator, so a batch is not
independently reproducible. Do the inverse-CDF draw yourself.

In [ ]:
def sample(logits, temps, top_ps, seeds, K=64):
  """One vectorized pass, and a per-request seed.

  torch.multinomial draws from ONE global generator, so two requests in
  the same batch are not independently reproducible. Do the inverse-CDF
  draw yourself, with one uniform per row from that row's own seed.
  """
  lg = logits / temps[:, None]
  srt, idx = torch.topk(lg, K, dim=-1)
  pr  = F.softmax(srt, dim=-1)
  cum = pr.cumsum(dim=-1)
  pr  = pr * ((cum - pr) < top_ps[:, None])
  pr  = pr / pr.sum(dim=-1, keepdim=True)

  # one uniform per row, each from its own generator
  u = 

  # inverse CDF: the first index where the cumulative mass passes u
  picked = 
  return idx.gather(1, picked[:, None]).squeeze(1)

lg, t, p = batch(8)
seeds = torch.arange(8)
a = sample(lg, t, p, seeds)
b = sample(lg, t, p, seeds)
print('same seeds give the same tokens:', torch.equal(a, b))
print('different seeds differ:         ',
      not torch.equal(a, sample(lg, t, p, seeds + 100)))

# Exercise 2: does it sample the right distribution?

Four tokens, known probabilities, twenty thousand draws. The histogram is the
test.

In [ ]:
# a distribution we know exactly, sampled many times
probs = torch.tensor([0.5, 0.3, 0.15, 0.05], device=dev)
logits = probs.log()[None, :].repeat(20000, 1)
temps  = torch.ones(20000, device=dev)
tops   = torch.ones(20000, device=dev)        # top_p = 1: no truncation
seeds  = torch.arange(20000)

drawn = 
empirical = 

print(f"{'token':>6} {'wanted':>8} {'got':>8}")
for i,(w,g) in enumerate(zip(probs.tolist(), empirical.tolist())):
  print(f'{i:>6} {w:>8.3f} {g:>8.3f}')
print(f'\nmax error {(empirical-probs).abs().max():.4f}')

# Exercise 3: is top-p exact?

Work out on paper which tokens `top_p = 0.9` keeps and what the renormalised
distribution over them is. Then check that is what you drew.

In [ ]:
target = 0.9
logits = probs.log()[None, :].repeat(20000, 1)
tops   = torch.full((20000,), target, device=dev)

drawn  = 
emp    = 

# work out by hand which tokens top_p=0.9 should keep, and what the
# renormalised distribution over them is
kept = 

print(f"{'token':>6} {'exact':>8} {'sampled':>9}")
for i,(w,g) in enumerate(zip(kept.tolist(), emp.tolist())):
  print(f'{i:>6} {w:>8.3f} {g:>9.3f}')
print(f'\nmax error {(emp-kept).abs().max():.4f}')

### Before you open the solution

1. Change `(cum - pr) < top_ps` to `cum < top_ps` and run Exercise 3 again.
   Which tokens survive now? Which result is correct?
2. Your Exercise 2 histogram must agree to approximately 1%. What does a
   missing renormalisation after the top-p mask do to it? Do you see the
   difference in the generated text?
3. One generator for each row is slow. For a batch of 256 that is 256 Python
   objects for each step. What do you keep between steps instead?